# Cross-cohort IVW meta-analysis of the seven replicated ME/CFS risk loci

This notebook reproduces the fixed-effect inverse-variance-weighted (IVW) meta-analysis and
forest plot reported in the manuscript.

Seven genomic risk loci reached a false discovery rate below 5% in the UK Biobank discovery
GWAS (UKB1) and replicated in at least one of two independent cohorts: a disjoint UK Biobank
cohort (UKB2) or the All of Us Research Program (AoU). Here the three per-cohort estimates for
each locus are pooled.

The three cohorts are **disjoint** so the three estimates for a locus are statistically 
independent and may be pooled directly. Every cohort was analysed with the same estimation
strategy targeting the same estimand. Between-cohort heterogeneity is
additionally reported as I² so that any inconsistency remains visible.

## Scale of the effects

Effects are **risk differences** (absolute change in ME/CFS risk). Risk
differences scale with the proportion of cases in the analysed sample, so all three cohorts were
downsampled to a common case fraction of ~1.11% (~89 controls per case) before estimation.
The estimates therefore share a scale and are pooled without rescaling.

## Requirements

```julia
import Pkg; Pkg.add(["CSV", "DataFrames", "CairoMakie"])
```

Input files are in `data/`. Running all cells writes `replication_meta.png` and
`replication_meta.pdf` to the working directory.

## 1. Configuration

`DIR` points at the folder holding the three per-cohort summary-statistic tables. The defaults
below read the copies bundled in `data/`; set the `META_DIR` environment variable to point
elsewhere.

In [ ]:
using CSV, DataFrames, Statistics, Printf, CairoMakie

# --- input files -------------------------------------------------------------
const DIR   = get(ENV, "META_DIR", "data")
const F_U1  = joinpath(DIR, "UKB1_Downsampled_Results.csv")   # discovery
const F_U2  = joinpath(DIR, "UKB2_Downsampled_Results.csv")   # replication 1
const F_AOU = joinpath(DIR, "aou_full_sumstats_support_filtered.csv")  # replication 2
const OUT   = get(ENV, "META_OUT", "replication_meta")       # output

const STUDY_ORDER = ["UKB1", "UKB2", "AoU"]   # order of rows within each locus block
const Z  = 1.959963984540054                  # normal quantile for a 95% interval
const FS = 20                                 # base font size (scales all text)

# --- figure palette (matches the other manuscript figures) -------------------
const BLUE = colorant"#1f6fb2"      # UKB1
const GREEN = colorant"#3ba99c"     # UKB2
const ORANGE = colorant"#e08a1e"    # AoU
const DIAMOND = colorant"#333333"   # pooled-estimate diamond
const I2_HI = colorant"#8a8a8a"     # I² >= 50%  (grey)
const I2_LO = colorant"#d1352b"     # I² <  50%  (red)
const STUDY_COLORS = Dict("UKB1" => BLUE, "UKB2" => GREEN, "AoU" => ORANGE)

## 2. Estimands: which genotype transition is pooled

TarGene estimates effects **per genotype transition** rather than assuming an additive dose
response:

| transition | meaning |
|---|---|
| `beta1` (β₁) | major homozygote → heterozygote |
| `beta2` (β₂) | heterozygote → minor homozygote |

Each locus is pooled on the transition that carries its association, declared explicitly in
`LOCI` below so the choice is auditable rather than inferred at run time. Six loci are pooled on
β₁. **rs261902 is pooled on β₂**: it is reported as a joint test and its discovery association is
driven by the β₂ (AG→AA) transition (discovery β₂ = −0.0043, p = 3.3×10⁻³, versus β₁ = −0.0017,
p = 1.1×10⁻²). Pooling it on β₁ would meta-analyse the wrong contrast.

`LOCI` is in genomic order and records, per locus: the rsID (the join key for the UK Biobank
tables), the AoU `variant_id`, which cohort the locus replicated in, and the transition.

In [ ]:
# column names for each transition in the TarGene output
const TRANS = Dict(
    "beta1" => (beta = "β₁", se = "se₁", ctrl = "control₁", cse = "case₁", short = "β₁"),
    "beta2" => (beta = "β₂", se = "se₂", ctrl = "control₂", cse = "case₂", short = "β₂"),
)

# The seven replicated loci, in genomic order.
#   (rsID, AoU GRCh38 variant_id, cohort the locus replicated in, transition pooled)
# AoU carries no rsID column and is reported on GRCh38, whereas the UK Biobank tables are on
# GRCh37 — so AoU rows are joined by variant_id via the mapping below, not by position.
const LOCI = [
    ("rs115186419", "3:186075329:T:C", "AoU",  "beta1"),
    ("rs73175505",  "8:5338479:C:T",   "UKB2", "beta1"),
    ("rs261902",    "12:32323793:A:G", "AoU",  "beta2"),   # joint locus -> β₂ (AG→AA)
    ("rs117553493", "13:99911262:C:T", "UKB2", "beta1"),
    ("rs72741654",  "15:61322139:C:T", "AoU",  "beta1"),
    ("rs74963073",  "16:10008215:C:T", "AoU",  "beta1"),
    ("rs76847656",  "16:28261692:T:C", "AoU",  "beta1"),
]

read_tsv(p) = CSV.read(p, DataFrame; delim = '\t', missingstring = ["", "NA", "NaN"])

## 3. Allele harmonisation

Effects are comparable across cohorts only if they count the **same allele**. TarGene labels each
transition by its two genotypes (e.g. `control₁ = "CC"`, `case₁ = "CT"`), so the effect allele is
whichever allele occurs more often in the case genotype than the control genotype.

`effect_allele` recovers it, and `build` (below) takes UKB1 as the anchor: any cohort whose
effect allele differs has its estimate **sign-flipped** so all three point in the same direction.
Without this step a cohort with reversed allele coding would pull the pooled estimate toward
zero and inflate I².

In [ ]:
"""
    effect_allele(control, case) -> Char

Return the allele that the transition adds — the one appearing more often in the case genotype
than in the control genotype (e.g. "CC" -> "CT" returns 'T').
"""
function effect_allele(control::AbstractString, cse::AbstractString)
    cc, kc = collect(control), collect(cse)
    for a in union(Set(cc), Set(kc))
        count(==(a), kc) > count(==(a), cc) && return a
    end
    return nothing
end

## 4. Fixed-effect IVW pooling

Each cohort is weighted by the reciprocal of its squared standard error, so more precise
estimates count for more:

$$w_i = \frac{1}{\mathrm{se}_i^2}, \qquad
\hat\beta = \frac{\sum_i w_i \beta_i}{\sum_i w_i}, \qquad
\mathrm{se}(\hat\beta) = \sqrt{\frac{1}{\sum_i w_i}}$$

Between-cohort heterogeneity uses Cochran's Q and I²:

$$Q = \sum_i w_i (\beta_i - \hat\beta)^2, \qquad
I^2 = \max\!\left(0, \frac{Q - \mathrm{df}}{Q}\right) \times 100\%, \qquad \mathrm{df} = k - 1$$

I² is the percentage of variability across cohorts attributable to genuine differences in effect
rather than sampling error. In the figure it is coloured grey at ≥ 50% and red below 50%.

In [ ]:
"""
    ivw(betas, ses) -> (est, lo, hi, I2)

Fixed-effect inverse-variance-weighted pooling of independent estimates, returning the pooled
effect, its 95% confidence bounds, and I² as a percentage.
"""
function ivw(betas, ses)
    w   = 1.0 ./ ses .^ 2                       # inverse-variance weights
    est = sum(w .* betas) / sum(w)              # pooled effect
    se  = sqrt(1 / sum(w))                      # pooled standard error
    Q   = sum(w .* (betas .- est) .^ 2)         # Cochran's Q
    df  = length(betas) - 1
    I2  = Q > 0 ? max(0.0, (Q - df) / Q) * 100 : 0.0
    return (est = est, lo = est - Z*se, hi = est + Z*se, I2 = I2)
end

## 5. Harmonise

For each locus: read the chosen transition from all three cohorts, harmonise the sign against the
UKB1 anchor, and pool. The returned records hold both the per-cohort estimates (plotted as
squares) and the pooled estimate (plotted as a diamond).

In [ ]:
"""
    build(idx) -> Vector{NamedTuple}

For every locus in `LOCI`, harmonise the three cohort estimates to a common effect allele and
pool them. `idx` maps cohort name -> Dict(rsID -> row).
"""
function build(idx)
    recs = NamedTuple[]
    for (rs, vid, rep, tid) in LOCI
        tr  = TRANS[tid]
        anc = idx["UKB1"][rs]                                   # UKB1 is the orientation anchor
        anchor_ea = effect_allele(String(anc[tr.ctrl]), String(anc[tr.cse]))
        trans = "$(anc[tr.ctrl])->$(anc[tr.cse])"               # e.g. "CC->CT", for the row label

        ps = NamedTuple[]
        for s in STUDY_ORDER
            r = idx[s][rs]
            # flip the sign if this cohort counts the other allele
            sgn = effect_allele(String(r[tr.ctrl]), String(r[tr.cse])) == anchor_ea ? 1.0 : -1.0
            b   = sgn * Float64(r[tr.beta])
            se  = Float64(r[tr.se])
            push!(ps, (study = s, beta = b, se = se, lo = b - Z*se, hi = b + Z*se))
        end

        m = ivw([p.beta for p in ps], [p.se for p in ps])
        push!(recs, (rs = rs, label = "$(rs)   $(trans)", rep = rep,
                     perstudy = ps, est = m.est, lo = m.lo, hi = m.hi, I2 = m.I2))
    end
    return recs
end

## 6. Forest plot

Three panels: locus and cohort labels on the left, the forest in the centre, and the numeric
estimates with I² on the right. Each cohort's marker area is proportional to its meta-analysis
weight (1/se²), so the visual size shows how much that cohort contributed. The pooled estimate is
the black diamond, whose width is its 95% confidence interval.

In [ ]:
function forest(recs)
    rowh, gap = 1.0, 0.9; y = 0.0
    lay = Dict{String,Any}()
    for r in recs
        yh = y; y -= 1.15rowh; ys = Float64[]
        for _ in r.perstudy; push!(ys, y); y -= rowh; end
        ysum = y; y -= rowh; y -= gap
        lay[r.rs] = (head = yh, studies = ys, sum = ysum)
    end
    ymin = y

    los = vcat([p.lo for r in recs for p in r.perstudy], [r.lo for r in recs])
    his = vcat([p.hi for r in recs for p in r.perstudy], [r.hi for r in recs])
    xlo, xhi = minimum(los), maximum(his); pad = 0.13*(xhi - xlo); xlo -= pad; xhi += pad
    ws = [p.se^-2 for r in recs for p in r.perstudy]; wmin, wmax = minimum(ws), maximum(ws)
    msize(se) = 14 + 26 * (se^-2 - wmin) / (wmax - wmin + eps())

    fig = Figure(size = (1600, max(1000, Int(round(-ymin*46)))), figure_padding = (8, 20, 12, 12))
    axL = Axis(fig[1, 1]); axF = Axis(fig[1, 2]); axR = Axis(fig[1, 3])
    # narrow label column removes the leftmost whitespace; forest gets the space,
    # and axR is a touch wider so I² doesn't collide with the CI text at this font.
    colsize!(fig.layout, 1, Relative(0.17)); colsize!(fig.layout, 2, Relative(0.55)); colsize!(fig.layout, 3, Relative(0.28))
    colgap!(fig.layout, 8)
    rowgap!(fig.layout, 8)
    for a in (axL, axR); hidedecorations!(a); hidespines!(a); xlims!(a, 0, 1); end
    hideydecorations!(axF); hidespines!(axF, :t, :r, :l)
    axF.xlabel = "Risk difference"; axF.xlabelsize = FS + 1; axF.xticklabelsize = FS - 2
    for a in (axL, axF, axR); ylims!(a, ymin - 0.6, 1.5); end
    xlims!(axF, xlo, xhi); vlines!(axF, [0.0]; color = (:gray, 0.65), linestyle = :dash, linewidth = 1.4)

    text!(axL, 0.0, 1.2; text = "Variant / Cohort", font = :bold, fontsize = FS + 1, align = (:left, :center))
    text!(axR, 0.0, 1.2; text = "RD [95% CI]",       font = :bold, fontsize = FS + 1, align = (:left, :center))
    text!(axR, 1.0, 1.2; text = "I²",                font = :bold, fontsize = FS + 1, align = (:right, :center))

    for r in recs
        L = lay[r.rs]
        text!(axL, 0.0, L.head; text = r.label, font = :bold, fontsize = FS + 1, align = (:left, :center))
        for (p, yy) in zip(r.perstudy, L.studies)
            col = STUDY_COLORS[p.study]
            lines!(axF, [p.lo, p.hi], [yy, yy]; color = col, linewidth = 3.2)
            scatter!(axF, [p.beta], [yy]; markersize = msize(p.se), color = col, marker = :rect)
            text!(axL, 0.10, yy; text = p.study, color = col, fontsize = FS, align = (:left, :center))
            text!(axR, 0.0, yy; text = @sprintf("%+.4f [%+.4f, %+.4f]", p.beta, p.lo, p.hi),
                  fontsize = FS - 4, color = colorant"#404040", align = (:left, :center))
        end
        yc, h = L.sum, 0.42
        poly!(axF, Point2f[(r.lo, yc), (r.est, yc + h), (r.hi, yc), (r.est, yc - h)]; color = DIAMOND)
        text!(axL, 0.10, yc; text = "Summary measure", font = :italic, fontsize = FS, align = (:left, :center))
        text!(axR, 0.0, yc; text = @sprintf("%+.4f [%+.4f, %+.4f]", r.est, r.lo, r.hi),
              fontsize = FS - 3, font = :bold, align = (:left, :center))
        text!(axR, 1.0, yc; text = @sprintf("%.0f%%", r.I2),
              color = r.I2 >= 50 ? I2_HI : I2_LO, font = :bold, fontsize = FS, align = (:right, :center))
    end

    save("$(OUT).png", fig; px_per_unit = 3)   # high-res raster
    save("$(OUT).pdf", fig)                     # vector
    @info "wrote $(OUT).png (3x) and $(OUT).pdf"
    return fig
end

## 7. Run

Reads the three tables, maps AoU `variant_id`s to rsIDs, builds the pooled records, prints them,
and renders the figure to `replication_meta.png` (3× raster) and `replication_meta.pdf` (vector).

In [ ]:
function main()
    u1 = read_tsv(F_U1); u2 = read_tsv(F_U2); aou = read_tsv(F_AOU)

    # AoU has no rsID column: map its GRCh38 variant_id to the rsID used by the other cohorts
    vid2rs = Dict(l[2] => l[1] for l in LOCI)
    aou.rs = [get(vid2rs, String(v), missing) for v in aou.variant_id]

    idx = Dict("UKB1" => Dict(String(r.rsID) => r for r in eachrow(u1)),
               "UKB2" => Dict(String(r.rsID) => r for r in eachrow(u2)),
               "AoU"  => Dict(String(r.rs)  => r for r in eachrow(aou) if !ismissing(r.rs)))

    recs = build(idx)
    for r in recs
        @info @sprintf("%-26s pooled=%+.5f [%+.5f, %+.5f]  I²=%.0f%%  rep=%s",
                       r.label, r.est, r.lo, r.hi, r.I2, r.rep)
    end
    forest(recs)
end

main()

## Expected output

Running the notebook end to end should reproduce the following pooled estimates:

| Locus | Transition | Pooled risk difference | 95% CI | I² | Replicated in |
|---|---|---|---|---|---|
| rs115186419 | TT→TC | −0.00467 | [−0.00629, −0.00305] | 33% | AoU |
| rs73175505 | CC→CT | −0.00504 | [−0.00651, −0.00356] | 50% | UKB2 |
| rs261902 | AG→AA | −0.00446 | [−0.00650, −0.00242] | 0% | AoU |
| rs117553493 | CC→CT | −0.00475 | [−0.00639, −0.00311] | 69% | UKB2 |
| rs72741654 | CC→CT | −0.00468 | [−0.00629, −0.00307] | 29% | AoU |
| rs74963073 | CC→CT | −0.00471 | [−0.00643, −0.00299] | 72% | AoU |
| rs76847656 | TT→TC | −0.00437 | [−0.00595, −0.00280] | 58% | AoU |

All pooled effects are negative, i.e. the counted (minor) allele is associated with lower ME/CFS
risk on the absolute risk-difference scale.

**Interpreting I² here.** Each pooled estimate combines only three cohorts, so I² is estimated
imprecisely and individual values should not be over-read. Note also that the discovery cohort
(UKB1) contributes the largest weight at every locus while also being the cohort in which these
loci were selected; pooled estimates therefore remain subject to the winner's curse and are
expected to be larger in magnitude than the true effects.